In [ ]:
import sys                      # Permet de configurer les chemins des imports
from pathlib import Path        # Permet de manipuler les chemins de fichiers

project_root = Path.cwd()       # Récupère le dossier de travail de Jupyter

if not (project_root / "scripts").is_dir():  # Si on n'est pas déjà à la racine
    project_root = project_root.parent.parent  # Remonte depuis notebooks/fine_tuning

assert (project_root / "scripts").is_dir(), "Vérifie le dossier de travail."

if str(project_root) not in sys.path:       # Évite d'ajouter deux fois le chemin
    sys.path.insert(0, str(project_root))   # Rend les scripts du projet importables

print("Racine du projet :", project_root)  # Affiche le chemin trouvé

In [ ]:
import torch                           # Bibliothèque utilisée pour entraîner le modèle
from transformers import set_seed      # Fonction qui fixe les graines aléatoires

SEED = 42                              # Même valeur pour reproduire nos expériences
set_seed(SEED)                         # Fixe le hasard de Python, NumPy et PyTorch

cuda_available = torch.cuda.is_available()  # Vérifie si PyTorch peut utiliser CUDA

print("Version PyTorch :", torch.__version__)  # Affiche la version installée
print("CUDA disponible :", cuda_available)     # Affiche True ou False

if cuda_available:                            # Si un GPU CUDA est accessible
    print("GPU :", torch.cuda.get_device_name(0))  # Affiche le nom du premier GPU

In [ ]:
# Chemin des annotations dans l'environnement d'origine
DATA_DIR = Path("/mnt/imported/data/sav-label-studio/Tasks-intention-transaction/OutputTasks")

assert DATA_DIR.is_dir(), f"Dossier introuvable : {DATA_DIR}"  # Vérifie le chemin

data_files = sorted(
    path for path in DATA_DIR.rglob("*") if path.is_file()
)  # Liste les fichiers du dossier et de ses sous-dossiers

print("Nombre de fichiers :", len(data_files))  # Compte les fichiers trouvés

for path in data_files[:5]:   # Prend les cinq premiers fichiers
    print(path.name)          # Affiche leur nom

In [ ]:
import json  # Permet de lire les fichiers JSON

assert data_files, "Aucun fichier trouvé."  # Vérifie que la liste n'est pas vide

example_path = data_files[0]  # Choisit le premier fichier de la liste

with example_path.open("r", encoding="utf-8") as file:  # Ouvre le fichier en lecture
    raw_example = json.load(file)                    # Charge son contenu en Python

print(
    json.dumps(raw_example, indent=2, ensure_ascii=False)
)  # Affiche le contenu lisiblement, en conservant les accents

In [ ]:
def get_choices(example, field_name):          # Recherche un champ d'annotation
    for annotation in example["result"]:      # Parcourt les annotations du message
        if annotation["from_name"] == field_name:  # Repère le champ demandé
            return annotation["value"].get("choices", [])  # Renvoie ses choix

    return []  # Renvoie une liste vide si le champ n'existe pas


text = raw_example["task"]["data"]["message"]  # Récupère le texte du message
intents = get_choices(raw_example, "intents")  # Récupère toutes ses intentions
intent_types = get_choices(raw_example, "type_intent")  # Récupère son type

print("Texte :", text)                        # Affiche le message
print("Intentions :", intents)                # Affiche la liste des intentions
print("Types d'intention :", intent_types)    # Affiche les choix du champ type

In [ ]:
def prepare_example(example):  # Transforme une annotation en un exemple simplifié
    if example["was_cancelled"]:  # Vérifie si l'annotation a été annulée
        return None               # Indique qu'on ne garde pas cet exemple

    review = get_choices(example, "review_decision")  # Récupère la décision de relecture

    if review != ["Accepter"]:  # Écarte les annotations non acceptées
        return None

    data = example["task"]["data"]  # Récupère les données du message
    intents = get_choices(example, "intents")  # Conserve toutes les intentions
    intent_types = get_choices(example, "type_intent")  # Récupère les types sélectionnés

    if not intents:  # Vérifie qu'au moins une intention est renseignée
        raise ValueError("Cet exemple accepté n'a aucune intention.")

    if len(intent_types) != 1:  # Vérifie qu'il y a exactement un type d'intention
        raise ValueError(f"Un seul type attendu, trouvé : {intent_types}")

    assimilated = get_choices(example, "Assimilated_by_Mode")  # Garde cette information pour le filtrage

    return {  # Construit le dictionnaire représentant notre exemple
        "text": data["message"],              # Texte donné au modèle
        "intents": intents,                   # Liste des intentions à prédire
        "type_intent": intent_types[0],       # Unique type à prédire
        "split": data["split"],               # Groupe d'origine : train ou test
        "Assimilated_by_Mode": assimilated[0] if assimilated else None,  # None si absent
    }


prepared_example = prepare_example(raw_example)  # Applique la fonction au fichier déjà lu
print(prepared_example)                         # Affiche le résultat

In [ ]:
def prepare_example(example):  # Transforme une annotation en un exemple simplifié
    if example["was_cancelled"]:  # Vérifie si l'annotation a été annulée
        return None               # Indique qu'on ne garde pas cet exemple

    review = get_choices(example, "review_decision")  # Récupère la décision de relecture

    if review != ["Accepter"]:  # Écarte les annotations non acceptées
        return None

    data = example["task"]["data"]  # Récupère les données du message
    intents = get_choices(example, "intents")  # Conserve toutes les intentions
    intent_types = get_choices(example, "type_intent")  # Récupère les types sélectionnés

    if not intents:  # Vérifie qu'au moins une intention est renseignée
        raise ValueError("Cet exemple accepté n'a aucune intention.")

    if len(intent_types) != 1:  # Vérifie qu'il y a exactement un type d'intention
        raise ValueError(f"Un seul type attendu, trouvé : {intent_types}")

    assimilated = get_choices(example, "Assimilated_by_Mode")  # Garde cette information pour le filtrage

    return {  # Construit le dictionnaire représentant notre exemple
        "text": data["message"],              # Texte donné au modèle
        "intents": intents,                   # Liste des intentions à prédire
        "type_intent": intent_types[0],       # Unique type à prédire
        "split": data["split"],               # Groupe d'origine : train ou test
        "Assimilated_by_Mode": assimilated[0] if assimilated else None,  # None si absent
    }


prepared_example = prepare_example(raw_example)  # Applique la fonction au fichier déjà lu
print(prepared_example)                         # Affiche le résultat

In [ ]:
examples = []        # Contiendra les exemples conservés
excluded_count = 0   # Compte les annotations annulées ou non acceptées

for path in data_files:  # Parcourt les chemins des fichiers trouvés précédemment
    with path.open("r", encoding="utf-8") as file:  # Ouvre le fichier courant
        raw_example = json.load(file)            # Charge son annotation JSON

    example = prepare_example(raw_example)  # Extrait les champs et vérifie l'annotation

    if example is None:         # Si la fonction a écarté cette annotation
        excluded_count += 1    # Ajoute 1 au compteur
        continue               # Passe directement au fichier suivant

    examples.append(example)   # Ajoute l'exemple conservé à notre liste

print("Exemples conservés :", len(examples))  # Affiche la taille de notre liste
print("Annotations écartées :", excluded_count)  # Affiche le nombre d'exclusions

assert examples, "Aucun exemple conservé : vérifie les fichiers et les filtres."

print(examples[0])  # Affiche le premier exemple conservé

In [ ]:
from collections import Counter  # Permet de compter les occurrences de chaque valeur

split_counts = Counter(example["split"] for example in examples)  # Compte les exemples par groupe

type_counts = Counter(
    example["type_intent"] for example in examples
)  # Compte les exemples pour chaque type d'intention

assimilation_counts = Counter(
    example["Assimilated_by_Mode"] for example in examples
)  # Compte les différentes valeurs de ce champ, y compris None

multilabel_count = sum(
    len(example["intents"]) > 1 for example in examples
)  # Compte les exemples qui possèdent plusieurs intentions

print("Répartition par groupe :", dict(split_counts))  # Affiche les effectifs train/test
print("Types d'intention :", dict(type_counts))        # Affiche les effectifs par type
print("Assimilation :", dict(assimilation_counts))     # Affiche les informations de filtrage
print("Exemples avec plusieurs intentions :", multilabel_count)
print("Nombre total d'exemples :", len(examples))

In [ ]:
EXCLUDE_ASSIMILATED = True  # Active l'exclusion des exemples marqués comme assimilés

if EXCLUDE_ASSIMILATED:  # Applique le filtre uniquement s'il est activé
    filtered_examples = [
        example for example in examples  # Parcourt les exemples chargés
        if example["Assimilated_by_Mode"] != "Assimilés par le modèle"  # Garde les autres
    ]
else:
    filtered_examples = examples.copy()  # Copie la liste sans exclure d'exemples

removed_count = len(examples) - len(filtered_examples)  # Calcule le nombre d'exclusions

print("Exemples retirés :", removed_count)          # Affiche l'effet du filtre
print("Exemples restants :", len(filtered_examples))  # Affiche la nouvelle taille

assert filtered_examples, "Le filtrage a supprimé tous les exemples."

In [ ]:
split_names = {example["split"] for example in filtered_examples}  # Récupère les valeurs distinctes

unexpected_splits = split_names - {"train", "test"}  # Repère les valeurs non prévues

if unexpected_splits:  # Arrête l'exécution si un groupe n'est pas reconnu
    raise ValueError(f"Valeurs de split inattendues : {unexpected_splits}")

train_examples = [
    example for example in filtered_examples
    if example["split"] == "train"  # Garde les exemples destinés à l'entraînement
]

test_examples = [
    example for example in filtered_examples
    if example["split"] == "test"  # Garde les exemples destinés au test
]

assert train_examples, "Le groupe train est vide."  # Vérifie la présence de données d'entraînement
assert test_examples, "Le groupe test est vide."    # Vérifie la présence de données de test

print("Exemples train :", len(train_examples))  # Affiche la taille du groupe train
print("Exemples test :", len(test_examples))    # Affiche la taille du groupe test

In [ ]:
from sklearn.model_selection import train_test_split  # Permet de séparer une liste en deux groupes

VALIDATION_RATIO = 0.20  # Réserve 20 % du train d'origine pour la validation

train_pool = [
    example for example in filtered_examples
    if example["split"] == "train"  # Repart toujours du train d'origine
]

train_examples, validation_examples = train_test_split(
    train_pool,                  # Données à répartir entre entraînement et validation
    test_size=VALIDATION_RATIO,   # Proportion réservée à la validation
    random_state=SEED,            # Rend la séparation reproductible
    shuffle=True,                # Mélange les exemples avant de les séparer
)

print("Entraînement :", len(train_examples))       # Exemples utilisés pour apprendre
print("Validation :", len(validation_examples))   # Exemples utilisés pour choisir le modèle
print("Test :", len(test_examples))               # Exemples réservés à l'évaluation finale

In [ ]:
from collections import Counter  # Compte les occurrences de chaque intention

def count_intents(examples):  # Calcule les effectifs pour un groupe d'exemples
    counts = Counter()       # Commence avec un compteur vide

    for example in examples:               # Parcourt les messages
        for intent in set(example["intents"]):  # Prend chaque intention du message une seule fois
            counts[intent] += 1            # Ajoute un exemple pour cette intention

    return counts  # Renvoie les effectifs obtenus


train_counts = count_intents(train_examples)            # Compte dans le train
validation_counts = count_intents(validation_examples)  # Compte dans la validation

all_intents = sorted(
    set(train_counts) | set(validation_counts)
)  # Réunit les intentions des deux groupes et les trie alphabétiquement

for intent in all_intents:  # Affiche les effectifs de chaque intention
    print(
        f"{intent} : "
        f"train={train_counts[intent]}, "
        f"validation={validation_counts[intent]}"
    )

In [ ]:
train_texts = {
    example["text"] for example in train_examples
}  # Récupère les textes distincts du train

validation_texts = {
    example["text"] for example in validation_examples
}  # Récupère les textes distincts de la validation

test_texts = {
    example["text"] for example in test_examples
}  # Récupère les textes distincts du test

train_validation_overlap = train_texts & validation_texts  # Textes communs au train et à la validation
train_test_overlap = train_texts & test_texts              # Textes communs au train et au test
validation_test_overlap = validation_texts & test_texts    # Textes communs à la validation et au test

print("Textes communs train / validation :", len(train_validation_overlap))
print("Textes communs train / test :", len(train_test_overlap))
print("Textes communs validation / test :", len(validation_test_overlap))